# NB13 — Ensemble Avanzado: 4 Levers hacia NDCG@10 ≥ 0.030

**Objetivo:** Superar NDCG@10 = 0.026026 (Ensemble RP3opt+EASE, NB11) apuntando a ≥ 0.030 (+15%).

## Auditoría pre-código
| Artefacto | Estado | Notas |
|---|---|---|
| `encoders/rp3beta_optimized.pkl` | ✅ Existe | α=0.75, β=0.30 (Optuna 50 trials NB11) |
| `encoders/final_model_v4.pkl` | ✅ Existe | MultiVAE state_dict + top_items_global |
| `encoders/ease_optimized.pkl` | ❌ Pendiente | Se crea en Lever 1 |
| Score matrices (RP3/EASE/VAE) | ❌ No guardadas | Se recalculan por usuario |
| `data/processed/interaction_matrix.csv` | ✅ Existe | Fuente canónica |

## Plan de 4 levers
- **Lever A:** EASE^R λ Optuna (λ=500 nunca fue tuneado; precomputa G una sola vez)
- **Lever B:** Ensemble 3-way (RP3opt + EASE-tuned + MultiVAE) con Optuna de pesos
- **Lever C:** Temporal Decay sobre RP3beta (ponderar interacciones por recencia)
- **Lever D:** Ensemble final con mejor componente RP3

**Disciplina de evaluación:**  
- `val_users` (~450, 15% estratificado) → hyperparameter search  
- `test_users_b` (~2550, 85%) → evaluación final UNA SOLA VEZ por modelo
- Split idéntico al de NB11 (seed=42, grupos cold/tepid/warm)

In [1]:
import sys
import time
import math
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from scipy.linalg import cho_factor, cho_solve
from sklearn.preprocessing import normalize as skl_normalize

import torch
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Rutas ────────────────────────────────────────────────────────────────────
HERE     = Path().resolve()
ROOT     = HERE.parent if (HERE.parent / "data").exists() else HERE
DATA_DIR = ROOT / "data" / "processed"
ENC_DIR  = ROOT / "encoders"
DOCS_DIR = ROOT / "docs"
SCRIPTS  = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))
DOCS_DIR.mkdir(exist_ok=True)

# ── Constantes globales (idénticas a NB11) ───────────────────────────────────
RANDOM_STATE  = 42
K_VALUES      = [5, 10, 20]
N_EVAL_USERS  = 3_000
CUTOFF_DATE   = pd.Timestamp("2015-08-22", tz="UTC")
EASE_TOP      = 20_000
EASE_REG_BASE = 500.0      # λ original de NB11 (nunca optimizado)

# Parámetros RP3beta óptimos de NB11
RP3_ALPHA_OPT = 0.75
RP3_BETA_OPT  = 0.30

# Optuna trials por lever
N_TRIALS_EASE     = 25   # Lever A — EASE λ
N_TRIALS_ENSEMBLE = 50   # Lever B — 3-way ensemble
N_TRIALS_FINAL    = 30   # Lever D — ensemble final
N_DECAY_RATES     = 6    # Lever C — temporal decay grid

print(f"Root     : {ROOT}")
print(f"Optuna   : {optuna.__version__}")
print(f"PyTorch  : {torch.__version__}")
print(f"EASE_TOP : {EASE_TOP}  λ_base: {EASE_REG_BASE}")
print(f"RP3opt   : alpha={RP3_ALPHA_OPT}  beta={RP3_BETA_OPT}")

Root     : C:\Users\Usuario\Desktop\HENRY\PROYECTO FINAL\EDA PRUEBA\nexus-recsys
Optuna   : 4.7.0
PyTorch  : 2.10.0+cpu
EASE_TOP : 20000  λ_base: 500.0
RP3opt   : alpha=0.75  beta=0.3


## 1 — Carga de datos, split temporal y val/test split (idéntico a NB11)

In [2]:
print("Cargando interaction_matrix.csv...")
t0 = time.time()
im  = pd.read_csv(DATA_DIR / "interaction_matrix.csv")
im["last_interaction_ts"] = pd.to_datetime(im["last_interaction_ts"], format="ISO8601", utc=True)
print(f"  IM: {im.shape}  [{time.time()-t0:.1f}s]")

# ── Split temporal (idéntico a NB09/NB11) ────────────────────────────────────
train_mask = im["last_interaction_ts"] < CUTOFF_DATE
train_df   = im[train_mask].copy()
test_df    = im[~train_mask].copy()

# Usuarios warm: tienen al menos 1 ítem en train Y 1 en test
warm_users = sorted(set(train_df["visitorid"].unique()) & set(test_df["visitorid"].unique()))

# Muestrear N_EVAL_USERS (seed=42, idéntico a NB09)
rng = np.random.default_rng(RANDOM_STATE)
eval_users = rng.choice(warm_users, size=min(N_EVAL_USERS, len(warm_users)), replace=False).tolist()

# Diccionarios de test
test_items_by_user  = test_df.groupby("visitorid")["itemid"].apply(set).to_dict()
train_items_by_user = train_df.groupby("visitorid")["itemid"].apply(set).to_dict()
test_tx_by_user     = (
    test_df[test_df["last_interaction_type"] == "transaction"]
    .groupby("visitorid")["itemid"].apply(set).to_dict()
)

all_items_global = sorted(im["itemid"].unique())
n_items_global   = len(all_items_global)
n_test_tx        = len(test_df[test_df["last_interaction_type"] == "transaction"])
baseline_conv    = n_test_tx / (len(set(test_df["visitorid"])) * n_items_global)

print(f"Train: {len(train_df):,}  Test: {len(test_df):,}")
print(f"Warm users total: {len(warm_users):,}")
print(f"Eval users:       {len(eval_users):,}")

# ── Val/Test split estratificado (IDÉNTICO a NB11) ───────────────────────────
activity_groups = []
for u in eval_users:
    cnt = len(train_items_by_user.get(u, set()))
    if   cnt == 1: activity_groups.append(0)   # cold
    elif cnt <= 4: activity_groups.append(1)   # tepid
    else:          activity_groups.append(2)   # warm
activity_groups = np.array(activity_groups)

rng_split = np.random.default_rng(RANDOM_STATE)
val_mask  = np.zeros(len(eval_users), dtype=bool)
for g in [0, 1, 2]:
    idx_g = np.where(activity_groups == g)[0]
    if len(idx_g) == 0:
        continue
    n_val  = max(1, int(len(idx_g) * 0.15))
    chosen = rng_split.choice(idx_g, size=n_val, replace=False)
    val_mask[chosen] = True

eval_arr     = np.array(eval_users)
val_users    = eval_arr[val_mask].tolist()       # ~450 usuarios (15%)
test_users_b = eval_arr[~val_mask].tolist()      # ~2550 usuarios (85%)

print(f"val_users    : {len(val_users):,}  (15% estratificado)")
print(f"test_users_b : {len(test_users_b):,}  (85% test definitivo)")
print(f"Grupos — cold:{(activity_groups==0).sum()}  tepid:{(activity_groups==1).sum()}  warm:{(activity_groups==2).sum()}")

Cargando interaction_matrix.csv...
  IM: (2145179, 7)  [6.8s]
Train: 1,763,782  Test: 381,397
Warm users total: 20,577
Eval users:       3,000
val_users    : 449  (15% estratificado)
test_users_b : 2,551  (85% test definitivo)
Grupos — cold:1724  tepid:894  warm:382


## 2 — Construcción de matrices y funciones base

In [3]:
# ── Construir R (CSR) sobre train ─────────────────────────────────────────────
all_train_users = sorted(train_df["visitorid"].unique())
all_train_items = sorted(train_df["itemid"].unique())
user2idx = {u: i for i, u in enumerate(all_train_users)}
item2idx = {it: i for i, it in enumerate(all_train_items)}
idx2item = {i: it for it, i in item2idx.items()}
n_u = len(all_train_users)
n_i = len(all_train_items)

rows_r = train_df["visitorid"].map(user2idx).values
cols_r = train_df["itemid"].map(item2idx).values
vals_r = train_df["interaction_strength"].values.astype(np.float32)
R = sp.csr_matrix((vals_r, (rows_r, cols_r)), shape=(n_u, n_i), dtype=np.float32)

item_pop      = np.asarray(R.sum(axis=0)).ravel()
item_pop_dict = {idx2item[i]: float(item_pop[i]) for i in range(n_i)}
n_total_train = float(R.sum())

print(f"R: {R.shape}  nnz={R.nnz:,}")

# ── Subespacio top-20K ítems (idéntico a NB11) ────────────────────────────────
top_items_idx = np.argpartition(item_pop, -EASE_TOP)[-EASE_TOP:]
top_items_idx = top_items_idx[np.argsort(item_pop[top_items_idx])[::-1]]
N_TOP         = len(top_items_idx)
top_items_global = [idx2item[i] for i in top_items_idx]

X_top_csr = R[:, top_items_idx].astype(np.float32).tocsr()  # (n_u, N_TOP)
pop_sub   = item_pop[top_items_idx].astype(np.float32)

print(f"X_top_csr: {X_top_csr.shape}  nnz={X_top_csr.nnz:,}")
print(f"pop_sub  : min={pop_sub.min():.0f}  max={pop_sub.max():.0f}")

R: (1158463, 215964)  nnz=1,763,782
X_top_csr: (1158463, 20000)  nnz=1,017,612
pop_sub  : min=27  max=2818


In [4]:
# ── Framework de evaluación (idéntico a NB11) ─────────────────────────────────
def ndcg(r, rel, k):
    r = r[:k]; dcg = sum((1/math.log2(i+2)) for i,x in enumerate(r) if x in rel)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(rel), k))); return dcg/idcg if idcg else 0.
def prec(r, rel, k):  return len(set(r[:k]) & rel) / k if k else 0.
def rec(r, rel, k):   return len(set(r[:k]) & rel) / len(rel) if rel else 0.
def ap(r, rel, k):
    if not rel: return 0.
    s, h = 0., 0
    for i, x in enumerate(r[:k]):
        if x in rel: h += 1; s += h / (i + 1)
    return s / min(len(rel), k)
def nov(flat, pd_, nt):
    return float(np.mean([-math.log2(pd_.get(x, 1) / nt + 1e-10) for x in flat])) if flat else 0.
def rev_k(r, tx, k): return len(set(r[:k]) & tx) / k if k else 0.
def ctr_k(r, ts, k): return len(set(r[:k]) & ts) / k if k else 0.

def evaluate(get_fn, evals, tst, tst_tx, pop_d, nt, cat_sz, bconv, ks=K_VALUES):
    acc = {k: {m: [] for m in "prnm"} for k in ks}
    for k in ks: acc[k]["r2"] = []; acc[k]["c"] = []
    seen = set(); ne = 0
    for uid in evals:
        ti = tst.get(uid, set())
        if not ti: continue
        tx = tst_tx.get(uid, set())
        mk = max(ks)
        try: recs = get_fn(uid, mk)
        except Exception: continue
        seen.update(recs); ne += 1
        for k in ks:
            acc[k]["p"].append(prec(recs, ti, k))
            acc[k]["r"].append(rec(recs,  ti, k))
            acc[k]["n"].append(ndcg(recs,  ti, k))
            acc[k]["m"].append(ap(recs,   ti, k))
            acc[k]["r2"].append(rev_k(recs, tx, k))
            acc[k]["c"].append(ctr_k(recs, ti, k))
    out = {"n_eval": ne}
    for k in ks:
        if not acc[k]["p"]: continue
        out[f"NDCG@{k}"]      = float(np.mean(acc[k]["n"]))
        out[f"Precision@{k}"] = float(np.mean(acc[k]["p"]))
        out[f"Recall@{k}"]    = float(np.mean(acc[k]["r"]))
        out[f"MAP@{k}"]       = float(np.mean(acc[k]["m"]))
        out[f"Revenue@{k}"]   = float(np.mean(acc[k]["r2"]))
        out[f"CTR@{k}"]       = float(np.mean(acc[k]["c"]))
        rv = out[f"Revenue@{k}"]
        out[f"ConvLift@{k}"]  = rv / bconv if bconv else 0.
    out["Coverage"] = len(seen) / cat_sz
    out["Novelty"]  = nov(list(seen), pop_d, nt)
    return out

# ── RP3beta builder ───────────────────────────────────────────────────────────
def build_rp3(alpha, beta, X_csr, pop_arr):
    P_ui = skl_normalize(X_csr.astype(np.float64), norm="l1", axis=1)
    P_it = skl_normalize(X_csr.T.tocsr().astype(np.float64), norm="l1", axis=1)
    pop_beta = np.power(pop_arr + 1e-10, beta)
    W_raw = P_it @ P_ui
    W = np.asarray(W_raw.todense() if hasattr(W_raw, "todense") else W_raw, dtype=np.float32)
    del W_raw
    np.power(W, alpha, out=W)
    W /= pop_beta[:, np.newaxis]
    np.fill_diagonal(W, 0.)
    return W

def make_get_rp3(W, X_csr, top_items_list):
    """Fabrica get_fn para RP3beta dado un W precomputado."""
    def get_fn(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        row = X_csr.getrow(ui)
        x_u = np.asarray(row.todense(), dtype=np.float32).ravel()
        sc  = x_u @ W
        sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [top_items_list[i] for i in top]
    return get_fn

# ── Min-max normalización por usuario (para ensembles) ────────────────────────
def minmax_norm(v):
    vmin, vmax = v.min(), v.max()
    rng = vmax - vmin
    return (v - vmin) / rng if rng > 1e-12 else np.zeros_like(v)

print("Funciones definidas: evaluate, build_rp3, make_get_rp3, minmax_norm")

Funciones definidas: evaluate, build_rp3, make_get_rp3, minmax_norm


## 3 — Carga de modelos existentes y baseline

In [5]:
# ── Cargar RP3beta optimizado (NB11) ─────────────────────────────────────────
print("Cargando rp3beta_optimized.pkl...")
t0 = time.time()
with open(ENC_DIR / "rp3beta_optimized.pkl", "rb") as f:
    art_rp3 = pickle.load(f)

W_rp3opt   = art_rp3["W_rp3"]             # (N_TOP, N_TOP) float32
# Usar top_items_global y user2idx de la sesión actual (reconstruidos desde IM)
# Los del pickle pueden ser consistentes; verificamos longitud
assert W_rp3opt.shape == (N_TOP, N_TOP), f"Shape mismatch: {W_rp3opt.shape}"
get_rp3opt = make_get_rp3(W_rp3opt, X_top_csr, top_items_global)
print(f"  RP3opt cargado  α={art_rp3['alpha']:.2f}  β={art_rp3['beta']:.2f}  "
      f"NDCG@10_test={art_rp3['ndcg10_test']:.5f}  [{time.time()-t0:.1f}s]")

Cargando rp3beta_optimized.pkl...
  RP3opt cargado  α=0.75  β=0.30  NDCG@10_test=0.02583  [7.9s]


In [6]:
# ── Cargar BPR (Matrix Factorization) desde final_model_v3.pkl ───────────────
# final_model_v4.pkl es solo un reporte (1.6 KB); pesos BPR están en v3
print("Cargando final_model_v3.pkl (extrayendo solo BPR)...")
t0 = time.time()
with open(ENC_DIR / "final_model_v3.pkl", "rb") as f:
    art_v3 = pickle.load(f)
print(f"  Claves: {list(art_v3.keys())}")

U_bpr   = art_v3["U_bpr"].astype(np.float32)   # (n_users, 64)
u2i_bpr = art_v3["user2idx"]   # visitorid -> row_idx in U_bpr
i2i_bpr = art_v3["item2idx"]   # itemid    -> col_idx in Vt_bpr
Vt_bpr  = art_v3["Vt_bpr"]
print(f"  U_bpr: {U_bpr.shape}   Vt_bpr: {Vt_bpr.shape}")

# Pre-extraer Vt para los top_items_global
bpr_cols  = np.array([i2i_bpr.get(it, -1) for it in top_items_global])
valid_bpr = bpr_cols >= 0
print(f"  Items top con factor BPR: {valid_bpr.sum():,} / {N_TOP}")

V_top_bpr = np.zeros((N_TOP, U_bpr.shape[1]), dtype=np.float32)
V_top_bpr[valid_bpr] = Vt_bpr[:, bpr_cols[valid_bpr]].T.astype(np.float32)

# Normalizar filas de V_top_bpr
norms = np.linalg.norm(V_top_bpr, axis=1, keepdims=True)
norms[norms < 1e-10] = 1.0
V_top_bpr /= norms

ndcg_bpr_saved = art_v3.get("ndcg10", "N/A")

# ── Liberar todo lo que no necesitamos de art_v3 ─────────────────────────────
del Vt_bpr
# Borrar explícitamente los arrays grandes dentro del dict antes de soltar art_v3
for _key in ["U_tdips", "Vt_tdips", "B_ease", "X_ease_csr", "s_tdips",
             "ncf_state_dict", "sasrec_state_dict", "idx2user", "idx2item",
             "U_bpr", "Vt_bpr"]:
    if _key in art_v3:
        del art_v3[_key]
del art_v3
import gc; gc.collect()
print(f"  V_top_bpr: {V_top_bpr.shape}  tiempo={time.time()-t0:.1f}s")
print(f"  BPR listo  NDCG@10_guardado={ndcg_bpr_saved}  Memoria liberada.")

def make_get_bpr(U, V_top, u2i, usr2idx_sess, train_X, top_items_list):
    def get_fn(uid, n):
        if uid not in u2i: return []
        ui_bpr = u2i[uid]
        sc = U[ui_bpr] @ V_top.T   # (N_TOP,)
        if uid in usr2idx_sess:
            ui_s = usr2idx_sess[uid]
            row  = train_X.getrow(ui_s)
            x_u  = np.asarray(row.todense(), dtype=np.float32).ravel()
            sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [top_items_list[i] for i in top]
    return get_fn

get_bpr = make_get_bpr(U_bpr, V_top_bpr, u2i_bpr, user2idx, X_top_csr, top_items_global)
print(f"  Scorer BPR listo  [{time.time()-t0:.1f}s]")


Cargando final_model_v3.pkl (extrayendo solo BPR)...
  Claves: ['model_name', 'ndcg10', 'all_metrics', 'hyperparams', 'U_tdips', 'Vt_tdips', 's_tdips', 'B_ease', 'X_ease_csr', 'top_items_global', 'top_items_idx', 'U_bpr', 'Vt_bpr', 'user2idx', 'item2idx', 'idx2item', 'idx2user', 'ensemble_weights', 'ncf_state_dict', 'ncf_config', 'sasrec_state_dict', 'sasrec_config']
  U_bpr: (1158463, 64)   Vt_bpr: (64, 215964)
  Items top con factor BPR: 20,000 / 20000
  V_top_bpr: (20000, 64)  tiempo=8.0s
  BPR listo  NDCG@10_guardado=0.02576  Memoria liberada.
  Scorer BPR listo  [8.0s]


In [7]:
# ── Precomputar G = X_top^T @ X_top (para Lever A Optuna) ────────────────────
# Se omite el baseline EASE(λ=500) para conservar memoria:
# G_for_optuna ocupa 1.6 GB float32; B_inv requiere otros 3.2 GB float64.
# No guardar G_dense por separado: usar G_for_optuna directamente.
import gc

print("Precomputando G = X_top^T @ X_top...")
t0 = time.time()
G_sparse = X_top_csr.T @ X_top_csr
G_for_optuna = np.asarray(G_sparse.todense(), dtype=np.float32)
del G_sparse
gc.collect()
print(f"  G_for_optuna: {G_for_optuna.shape}  dtype={G_for_optuna.dtype}  [{time.time()-t0:.1f}s]")

diag_idx = np.arange(N_TOP)

def make_get_ease(B):
    """Fabrica get_fn para EASE^R dado B precomputado."""
    def get_fn(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        row = X_top_csr.getrow(ui)
        x_u = np.asarray(row.todense(), dtype=np.float32).ravel()
        sc  = x_u @ B
        sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [top_items_global[i] for i in top]
    return get_fn

# NDCG@10 de referencia EASE(λ=500) — reportado en NB11
NDCG_EASE_BASE_NB11 = 0.024766   # EASE(λ=500) test NB11
print(f"G listo. Baseline EASE(λ=500) NB11 conocido: NDCG@10={NDCG_EASE_BASE_NB11:.5f}")
print("(baseline EASE no recomputado — se evita uso de 4 GB extra de RAM)")


Precomputando G = X_top^T @ X_top...
  G_for_optuna: (20000, 20000)  dtype=float32  [1.4s]
G listo. Baseline EASE(λ=500) NB11 conocido: NDCG@10=0.02477
(baseline EASE no recomputado — se evita uso de 4 GB extra de RAM)


In [8]:
# ── Evaluación baseline sobre val_users ──────────────────────────────────────
print(f"Evaluando modelos base sobre {len(val_users)} val_users...")

t0 = time.time()
m_rp3_val = evaluate(
    get_rp3opt, val_users,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  RP3opt   NDCG@10={m_rp3_val['NDCG@10']:.5f}  [{time.time()-t0:.1f}s]")

t0 = time.time()
m_bpr_val = evaluate(
    get_bpr, val_users,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  BPR-MF   NDCG@10={m_bpr_val['NDCG@10']:.5f}  [{time.time()-t0:.1f}s]")

ndcg_rp3_val  = m_rp3_val["NDCG@10"]
ndcg_ease_val = NDCG_EASE_BASE_NB11  # referencia NB11 — no se recomputa para ahorrar RAM
ndcg_bpr_val  = m_bpr_val["NDCG@10"]
print(f"\nBaseline val — RP3:{ndcg_rp3_val:.5f}  BPR:{ndcg_bpr_val:.5f}")
print(f"  EASE val ref (NB11): {ndcg_ease_val:.5f}  (no recomputado — ahorro de 4 GB RAM)")


Evaluando modelos base sobre 449 val_users...
  RP3opt   NDCG@10=0.01925  [54.5s]
  BPR-MF   NDCG@10=0.00833  [0.2s]

Baseline val — RP3:0.01925  BPR:0.00833
  EASE val ref (NB11): 0.02477  (no recomputado — ahorro de 4 GB RAM)


## Lever A — EASE^R λ Optuna

El λ=500 de NB11 nunca fue optimizado para este dataset.  
**Truco de eficiencia:** G = X^T @ X ya está precomputado; cada trial sólo recomputará inv(G + λI).

In [9]:
print(f"=" * 60)
print(f"LEVER A — EASE^R λ Optuna ({N_TRIALS_EASE} trials)")
print(f"  G precomputado: {G_for_optuna.shape}  search λ en [10, 5000] log-scale")
print(f"  Evaluación sobre {len(val_users)} val_users")
print(f"=" * 60)

diag_idx = np.arange(N_TOP)

def objective_ease(trial):
    lam = trial.suggest_float("lambda", 10.0, 5000.0, log=True)
    G_reg = G_for_optuna.copy()
    G_reg[diag_idx, diag_idx] += lam
    B_inv_ = np.linalg.inv(G_reg)
    del G_reg
    d_inv = np.diag(B_inv_).copy()
    B = -(B_inv_ / d_inv[np.newaxis, :]).astype(np.float32)
    np.fill_diagonal(B, 0.)
    del B_inv_
    get_fn = make_get_ease(B)
    m = evaluate(
        get_fn, val_users,
        test_items_by_user, test_tx_by_user,
        item_pop_dict, n_total_train, n_items_global, baseline_conv
    )
    return m.get("NDCG@10", 0.)

t_opt = time.time()
study_ease = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_ease.optimize(objective_ease, n_trials=N_TRIALS_EASE, show_progress_bar=True)
print(f"\nOptuna EASE finalizado en {time.time()-t_opt:.0f}s")

# Tabla de trials
trials_ease_df = pd.DataFrame([
    {"Rank": 0, "lambda": t.params["lambda"], "NDCG@10 val": t.value}
    for t in study_ease.trials if t.value is not None
]).sort_values("NDCG@10 val", ascending=False).reset_index(drop=True)
trials_ease_df["Rank"] = range(1, len(trials_ease_df) + 1)
trials_ease_df["lambda"] = trials_ease_df["lambda"].round(1)
trials_ease_df["NDCG@10 val"] = trials_ease_df["NDCG@10 val"].round(6)
print("\nTop 10 trials EASE^R λ:")
print(trials_ease_df.head(10).to_string(index=False))

best_lambda = study_ease.best_params["lambda"]
best_ease_val = study_ease.best_value
delta_ease = (best_ease_val - ndcg_ease_val) / ndcg_ease_val * 100
print(f"\nλ óptimo: {best_lambda:.1f}  (λ_base={EASE_REG_BASE})")
print(f"NDCG@10 val: ref_NB11={ndcg_ease_val:.5f} → opt={best_ease_val:.5f}  ({'+' if delta_ease>=0 else ''}{delta_ease:.1f}%)")

LEVER A — EASE^R λ Optuna (25 trials)
  G precomputado: (20000, 20000)  search λ en [10, 5000] log-scale
  Evaluación sobre 449 val_users


  0%|          | 0/25 [00:00<?, ?it/s]


Optuna EASE finalizado en 4029s

Top 10 trials EASE^R λ:
 Rank  lambda  NDCG@10 val
    1    14.3     0.014739
    2    10.9     0.014739
    3    15.8     0.014739
    4    16.0     0.014739
    5    10.5     0.014739
    6    17.8     0.014739
    7    10.5     0.014739
    8    10.1     0.014739
    9    11.4     0.014739
   10    30.1     0.014723

λ óptimo: 14.3  (λ_base=500.0)
NDCG@10 val: ref_NB11=0.02477 → opt=0.01474  (-40.5%)


In [10]:
# ── Lever A — Evaluación FINAL sobre test (UNA SOLA VEZ) ─────────────────────
print(f"Construyendo EASE^R(λ={best_lambda:.1f})...")
t0 = time.time()
G_reg_opt = G_for_optuna.copy()
G_reg_opt[diag_idx, diag_idx] += best_lambda
B_inv_opt = np.linalg.inv(G_reg_opt)
del G_reg_opt
d_inv_opt  = np.diag(B_inv_opt).copy()
B_ease_opt = -(B_inv_opt / d_inv_opt[np.newaxis, :]).astype(np.float32)
np.fill_diagonal(B_ease_opt, 0.)
del B_inv_opt
get_ease_opt = make_get_ease(B_ease_opt)
print(f"  Construido en {time.time()-t0:.1f}s")

print(f"Evaluando EASE^R(λ={best_lambda:.1f}) sobre test ({len(test_users_b)})...")
t0 = time.time()
m_ease_opt_test = evaluate(
    get_ease_opt, test_users_b,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  [{time.time()-t0:.1f}s]")

# Usar referencia NB11 como baseline EASE (get_ease_base no disponible — ahorro RAM)
ndcg_ease_base_test = NDCG_EASE_BASE_NB11   # EASE(λ=500) test reportado en NB11
ndcg_ease_opt_test  = m_ease_opt_test["NDCG@10"]
delta_a = (ndcg_ease_opt_test - ndcg_ease_base_test) / ndcg_ease_base_test * 100

print(f"\nLEVER A — Resultados test:")
print(f"  EASE(λ={EASE_REG_BASE}) NB11 ref: NDCG@10={ndcg_ease_base_test:.5f}")
print(f"  EASE(λ={best_lambda:.1f}) opt:   NDCG@10={ndcg_ease_opt_test:.5f}  ({'+' if delta_a>=0 else ''}{delta_a:.1f}%)")

# Guardar ease_optimized.pkl
artifact_ease = {
    "model_name":   f"EASE^R optimizado (λ={best_lambda:.1f})",
    "lambda":        best_lambda,
    "ndcg10_test":   ndcg_ease_opt_test,
    "ndcg10_val":    best_ease_val,
    "B_ease":        B_ease_opt,
    "top_items_global": top_items_global,
    "top_items_idx":    top_items_idx,
    "user2idx":         user2idx,
    "optuna_best_params": study_ease.best_params,
    "optuna_n_trials":    N_TRIALS_EASE,
}
ease_out = ENC_DIR / "ease_optimized.pkl"
with open(ease_out, "wb") as f:
    pickle.dump(artifact_ease, f, protocol=4)
import os
print(f"Guardado: {ease_out}  ({os.path.getsize(ease_out)/1e6:.1f} MB)")

Construyendo EASE^R(λ=14.3)...
  Construido en 120.9s
Evaluando EASE^R(λ=14.3) sobre test (2551)...
  [172.2s]

LEVER A — Resultados test:
  EASE(λ=500.0) NB11 ref: NDCG@10=0.02477
  EASE(λ=14.3) opt:   NDCG@10=0.02091  (-15.6%)
Guardado: C:\Users\Usuario\Desktop\HENRY\PROYECTO FINAL\EDA PRUEBA\nexus-recsys\encoders\ease_optimized.pkl  (1628.2 MB)


## Lever B — Ensemble 3-way (RP3opt + EASE-tuned + BPR-MF)

Optuna sobre pesos w1 (RP3), w2 (EASE), w3=1−w1−w2 (BPR),  
con restricción w1+w2 ≤ 1 y w1,w2,w3 ≥ 0.  
BPR es el modelo de Matrix Factorization (64 dimensiones) disponible en `final_model_v3.pkl`.

In [11]:
print("=" * 60)
print(f"LEVER B — Ensemble 3-way Optuna ({N_TRIALS_ENSEMBLE} trials)")
print(f"  Modelos: RP3opt | EASE(λ={best_lambda:.1f}) | BPR-MF")
print(f"  Evaluación sobre {len(val_users)} val_users")
print("=" * 60)

def make_3way_ensemble(w1, w2, W_rp3_, B_ease_, X_rp3_):
    """Ensemble 3-way: RP3 (w1) + EASE (w2) + BPR (w3=1-w1-w2)."""
    w3 = 1.0 - w1 - w2
    def get_fn(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        row = X_rp3_.getrow(ui)
        x_u = np.asarray(row.todense(), dtype=np.float32).ravel()

        sc_rp3  = minmax_norm(x_u @ W_rp3_)
        sc_ease = minmax_norm(x_u @ B_ease_)

        # BPR score
        if uid in u2i_bpr:
            sc_bpr = minmax_norm(U_bpr[u2i_bpr[uid]] @ V_top_bpr.T)
        else:
            sc_bpr = np.zeros(N_TOP, dtype=np.float32)

        sc = w1 * sc_rp3 + w2 * sc_ease + w3 * sc_bpr
        sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [top_items_global[i] for i in top]
    return get_fn

def objective_ensemble(trial):
    w1 = trial.suggest_float("w_rp3",  0.30, 0.80)
    w2 = trial.suggest_float("w_ease", 0.05, 0.40)
    if w1 + w2 > 1.0:
        raise optuna.exceptions.TrialPruned()
    get_fn = make_3way_ensemble(w1, w2, W_rp3opt, B_ease_opt, X_top_csr)
    m = evaluate(
        get_fn, val_users,
        test_items_by_user, test_tx_by_user,
        item_pop_dict, n_total_train, n_items_global, baseline_conv
    )
    return m.get("NDCG@10", 0.)

t_opt = time.time()
study_ens = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_ens.optimize(objective_ensemble, n_trials=N_TRIALS_ENSEMBLE, show_progress_bar=True)
print(f"\nOptuna ensemble finalizado en {time.time()-t_opt:.0f}s")

trials_ens_df = pd.DataFrame([
    {"Rank": 0, "w_rp3": t.params["w_rp3"], "w_ease": t.params["w_ease"],
     "w_bpr": 1-t.params["w_rp3"]-t.params["w_ease"],
     "NDCG@10 val": t.value}
    for t in study_ens.trials if t.value is not None and t.state.name == "COMPLETE"
]).sort_values("NDCG@10 val", ascending=False).reset_index(drop=True)
trials_ens_df["Rank"] = range(1, len(trials_ens_df) + 1)
for col in ["w_rp3", "w_ease", "w_bpr", "NDCG@10 val"]:
    trials_ens_df[col] = trials_ens_df[col].round(4)
print("Top 10 trials ensemble 3-way:")
print(trials_ens_df.head(10).to_string(index=False))

best_w1      = study_ens.best_params["w_rp3"]
best_w2      = study_ens.best_params["w_ease"]
best_w3      = 1.0 - best_w1 - best_w2
best_ens_val = study_ens.best_value
print(f"\nPesos óptimos: w_rp3={best_w1:.3f}  w_ease={best_w2:.3f}  w_bpr={best_w3:.3f}")
print(f"NDCG@10 val: {best_ens_val:.5f}")


LEVER B — Ensemble 3-way Optuna (50 trials)
  Modelos: RP3opt | EASE(λ=14.3) | BPR-MF
  Evaluación sobre 449 val_users


  0%|          | 0/50 [00:00<?, ?it/s]


Optuna ensemble finalizado en 4076s
Top 10 trials ensemble 3-way:
 Rank  w_rp3  w_ease  w_bpr  NDCG@10 val
    1 0.7126  0.0763 0.2112       0.0194
    2 0.7000  0.0676 0.2324       0.0194
    3 0.6962  0.0811 0.2226       0.0194
    4 0.7005  0.0732 0.2263       0.0194
    5 0.6785  0.0822 0.2393       0.0193
    6 0.6932  0.0722 0.2346       0.0193
    7 0.6891  0.0776 0.2333       0.0193
    8 0.6722  0.0908 0.2370       0.0193
    9 0.6747  0.0915 0.2337       0.0193
   10 0.6628  0.0934 0.2438       0.0193

Pesos óptimos: w_rp3=0.713  w_ease=0.076  w_bpr=0.211
NDCG@10 val: 0.01941


In [12]:
# ── Lever B — Evaluación FINAL en test (UNA SOLA VEZ) ────────────────────────
print(f"Evaluando ensemble 3-way (RP3+EASE+BPR) en test ({len(test_users_b)})...")
t0 = time.time()
get_ens3 = make_3way_ensemble(best_w1, best_w2, W_rp3opt, B_ease_opt, X_top_csr)
m_ens3_test = evaluate(
    get_ens3, test_users_b,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  [{time.time()-t0:.1f}s]")

ndcg_ens3_test = m_ens3_test["NDCG@10"]
NDCG_NB11_ENS2 = 0.026026
delta_b = (ndcg_ens3_test - NDCG_NB11_ENS2) / NDCG_NB11_ENS2 * 100
print(f"\nLEVER B — Resultados test:")
print(f"  Ensemble 3-way (RP3+EASE+BPR): NDCG@10={ndcg_ens3_test:.5f}")
print(f"  vs NB11 Ens2 (ref)            : NDCG@10={NDCG_NB11_ENS2:.5f}  ({'+' if delta_b>=0 else ''}{delta_b:.1f}%)")


Evaluando ensemble 3-way (RP3+EASE+BPR) en test (2551)...
  [483.0s]

LEVER B — Resultados test:
  Ensemble 3-way (RP3+EASE+BPR): NDCG@10=0.02498
  vs NB11 Ens2 (ref)            : NDCG@10=0.02603  (-4.0%)


## Lever C — Temporal Decay sobre RP3beta

Ponderar interacciones por **recencia**: `w_i = exp(-decay_rate × dias_antes_cutoff)`.  
Grilla de 6 `decay_rate` en `[0.0001, 0.05]`. Reconstruye R_td → X_top_td → RP3beta(α,β).

In [13]:
print(f"=" * 60)
print(f"LEVER C — Temporal Decay RP3beta ({N_DECAY_RATES} valores)")
print(f"  alpha={RP3_ALPHA_OPT}  beta={RP3_BETA_OPT}  (óptimos de NB11)")
print(f"=" * 60)

# Campo de timestamp para calcular días al cutoff
train_df_td = train_df.copy()
train_df_td["days_before"] = (
    (CUTOFF_DATE - train_df_td["last_interaction_ts"]).dt.total_seconds() / 86400.0
).clip(lower=0.0)

decay_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
results_decay = []

print(f"  {'decay_rate':>12}  {'NDCG@10 val':>12}  {'vs RP3opt val':>14}  {'time (s)':>10}")
print("-" * 55)

for dr in decay_rates:
    t0 = time.time()

    # Pesos de recencia
    decay_weights = np.exp(-dr * train_df_td["days_before"].values)
    # Ponderar interaction_strength por decay
    vals_td = (train_df_td["interaction_strength"].values.astype(np.float32)
               * decay_weights.astype(np.float32))

    rows_td = train_df_td["visitorid"].map(user2idx).values
    cols_td = train_df_td["itemid"].map(item2idx).values
    R_td = sp.csr_matrix((vals_td, (rows_td, cols_td)), shape=(n_u, n_i), dtype=np.float32)

    # Usar los mismos top_items_idx (definidos por popularidad original)
    X_top_td = R_td[:, top_items_idx].astype(np.float32).tocsr()

    # Popularidad en R_td para RP3
    pop_td = np.asarray(R_td.sum(axis=0)).ravel()[top_items_idx].astype(np.float32)

    W_td = build_rp3(RP3_ALPHA_OPT, RP3_BETA_OPT, X_top_td, pop_td)
    # Nota: make_get_rp3 usa user2idx global y X_top_td para scores
    get_rp3_td = make_get_rp3(W_td, X_top_td, top_items_global)

    m = evaluate(
        get_rp3_td, val_users,
        test_items_by_user, test_tx_by_user,
        item_pop_dict, n_total_train, n_items_global, baseline_conv
    )
    elapsed = time.time() - t0
    ndcg_td = m.get("NDCG@10", 0.)
    delta_vs_rp3 = (ndcg_td - ndcg_rp3_val) / ndcg_rp3_val * 100

    results_decay.append({
        "decay_rate": dr,
        "NDCG@10 val": ndcg_td,
        "delta_vs_rp3_val_%": round(delta_vs_rp3, 2),
        "W_td": W_td,
        "X_top_td": X_top_td,
        "time_s": round(elapsed, 1),
    })
    sign = "+" if delta_vs_rp3 >= 0 else ""
    print(f"  {dr:12.4f}  {ndcg_td:12.5f}  {sign}{delta_vs_rp3:>12.1f}%  {elapsed:9.0f}s")

df_decay = pd.DataFrame([{k: v for k, v in r.items() if k not in ("W_td", "X_top_td")} for r in results_decay])
best_decay_row   = df_decay.loc[df_decay["NDCG@10 val"].idxmax()]
best_decay_rate  = float(best_decay_row["decay_rate"])
best_decay_val   = float(best_decay_row["NDCG@10 val"])
print(f"\nMejor decay_rate: {best_decay_rate}  NDCG@10 val={best_decay_val:.5f}")
print(f"RP3opt sin decay : {ndcg_rp3_val:.5f}  → mejora: {best_decay_val - ndcg_rp3_val:+.5f}")

LEVER C — Temporal Decay RP3beta (6 valores)
  alpha=0.75  beta=0.3  (óptimos de NB11)
    decay_rate   NDCG@10 val   vs RP3opt val    time (s)
-------------------------------------------------------
        0.0001       0.01755          -8.8%         37s
        0.0005       0.01755          -8.8%         38s
        0.0010       0.01714         -11.0%         36s
        0.0050       0.01760          -8.6%         38s
        0.0100       0.01786          -7.2%         38s
        0.0500       0.01478         -23.2%         37s

Mejor decay_rate: 0.01  NDCG@10 val=0.01786
RP3opt sin decay : 0.01925  → mejora: -0.00139


In [14]:
# ── Lever C — Evaluación FINAL en test (UNA SOLA VEZ con mejor decay_rate) ───
best_td_entry = next(r for r in results_decay if r["decay_rate"] == best_decay_rate)
W_best_td     = best_td_entry["W_td"]
X_top_td_best = best_td_entry["X_top_td"]

get_rp3_td_best = make_get_rp3(W_best_td, X_top_td_best, top_items_global)

print(f"Evaluando RP3+TD(decay={best_decay_rate}) en test ({len(test_users_b)})...")
t0 = time.time()
m_rp3_td_test = evaluate(
    get_rp3_td_best, test_users_b,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  [{time.time()-t0:.1f}s]")

# Referencia RP3opt en test
print(f"Evaluando RP3opt (sin TD) en test...")
t0 = time.time()
m_rp3opt_test = evaluate(
    get_rp3opt, test_users_b,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  [{time.time()-t0:.1f}s]")

ndcg_rp3opt_test = m_rp3opt_test["NDCG@10"]
ndcg_rp3_td_test = m_rp3_td_test["NDCG@10"]
delta_c = (ndcg_rp3_td_test - ndcg_rp3opt_test) / ndcg_rp3opt_test * 100

print(f"\nLEVER C — Resultados test:")
print(f"  RP3opt (sin TD)   : NDCG@10={ndcg_rp3opt_test:.5f}")
print(f"  RP3+TD({best_decay_rate}) : NDCG@10={ndcg_rp3_td_test:.5f}  ({'+' if delta_c>=0 else ''}{delta_c:.1f}%)")

# ¿Usar RP3+TD o RP3opt como componente RP3 para Lever D?
use_td_in_final = ndcg_rp3_td_test > ndcg_rp3opt_test
print(f"\n→ RP3+TD mejora RP3opt: {use_td_in_final}")
if use_td_in_final:
    W_rp3_final  = W_best_td
    X_rp3_final  = X_top_td_best
    rp3_label    = f"RP3+TD(d={best_decay_rate})"
    ndcg_rp3_final_test = ndcg_rp3_td_test
else:
    W_rp3_final  = W_rp3opt
    X_rp3_final  = X_top_csr
    rp3_label    = "RP3opt"
    ndcg_rp3_final_test = ndcg_rp3opt_test
print(f"→ Componente RP3 para Lever D: {rp3_label}  NDCG@10={ndcg_rp3_final_test:.5f}")

Evaluando RP3+TD(decay=0.01) en test (2551)...
  [175.7s]
Evaluando RP3opt (sin TD) en test...
  [317.8s]

LEVER C — Resultados test:
  RP3opt (sin TD)   : NDCG@10=0.02583
  RP3+TD(0.01) : NDCG@10=0.02859  (+10.7%)

→ RP3+TD mejora RP3opt: True
→ Componente RP3 para Lever D: RP3+TD(d=0.01)  NDCG@10=0.02859


## Lever D — Ensemble final con mejor componente RP3

In [15]:
print("=" * 60)
print(f"LEVER D — Ensemble final ({N_TRIALS_FINAL} trials)")
print(f"  RP3: {rp3_label} | EASE(λ={best_lambda:.1f}) | BPR-MF")
print("=" * 60)

def make_final_ensemble(w1, w2, W_rp3_, B_ease_, X_rp3_):
    """Ensemble final: mejor RP3 (w1) + EASE-opt (w2) + BPR (w3=1-w1-w2)."""
    w3 = 1.0 - w1 - w2
    def get_fn(uid, n):
        if uid not in user2idx: return []
        ui   = user2idx[uid]
        row  = X_rp3_.getrow(ui)
        x_rp3 = np.asarray(row.todense(), dtype=np.float32).ravel()
        row_e = X_top_csr.getrow(ui)
        x_ease = np.asarray(row_e.todense(), dtype=np.float32).ravel()

        sc_rp3  = minmax_norm(x_rp3  @ W_rp3_)
        sc_ease = minmax_norm(x_ease @ B_ease_)

        if uid in u2i_bpr:
            sc_bpr = minmax_norm(U_bpr[u2i_bpr[uid]] @ V_top_bpr.T)
        else:
            sc_bpr = np.zeros(N_TOP, dtype=np.float32)

        sc = w1 * sc_rp3 + w2 * sc_ease + w3 * sc_bpr
        seen_mask = (x_rp3 > 0) | (x_ease > 0)
        sc[seen_mask] = -1.
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [top_items_global[i] for i in top]
    return get_fn

def objective_final(trial):
    w1 = trial.suggest_float("w_rp3",  0.20, 0.80)
    w2 = trial.suggest_float("w_ease", 0.05, 0.50)
    if w1 + w2 > 1.0:
        raise optuna.exceptions.TrialPruned()
    get_fn = make_final_ensemble(w1, w2, W_rp3_final, B_ease_opt, X_rp3_final)
    m = evaluate(
        get_fn, val_users,
        test_items_by_user, test_tx_by_user,
        item_pop_dict, n_total_train, n_items_global, baseline_conv
    )
    return m.get("NDCG@10", 0.)

t_opt = time.time()
study_final = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_final.optimize(objective_final, n_trials=N_TRIALS_FINAL, show_progress_bar=True)
print(f"\nOptuna final finalizado en {time.time()-t_opt:.0f}s")

trials_final_df = pd.DataFrame([
    {"Rank": 0, "w_rp3": t.params["w_rp3"], "w_ease": t.params["w_ease"],
     "w_bpr": 1-t.params["w_rp3"]-t.params["w_ease"],
     "NDCG@10 val": t.value}
    for t in study_final.trials if t.value is not None and t.state.name == "COMPLETE"
]).sort_values("NDCG@10 val", ascending=False).reset_index(drop=True)
trials_final_df["Rank"] = range(1, len(trials_final_df) + 1)
for col in ["w_rp3", "w_ease", "w_bpr", "NDCG@10 val"]:
    trials_final_df[col] = trials_final_df[col].round(4)
print("Top 10 trials ensemble final:")
print(trials_final_df.head(10).to_string(index=False))

final_w1  = study_final.best_params["w_rp3"]
final_w2  = study_final.best_params["w_ease"]
final_w3  = 1.0 - final_w1 - final_w2
final_val = study_final.best_value
print(f"\nPesos óptimos finales: w_rp3={final_w1:.3f}  w_ease={final_w2:.3f}  w_bpr={final_w3:.3f}")
print(f"NDCG@10 val: {final_val:.5f}")


LEVER D — Ensemble final (30 trials)
  RP3: RP3+TD(d=0.01) | EASE(λ=14.3) | BPR-MF


  0%|          | 0/30 [00:00<?, ?it/s]


Optuna final finalizado en 1814s
Top 10 trials ensemble final:
 Rank  w_rp3  w_ease  w_bpr  NDCG@10 val
    1 0.7946  0.0674 0.1380       0.0178
    2 0.7996  0.0826 0.1178       0.0178
    3 0.7960  0.0852 0.1188       0.0178
    4 0.7986  0.0588 0.1426       0.0177
    5 0.7928  0.0526 0.1546       0.0177
    6 0.7868  0.0527 0.1605       0.0177
    7 0.7817  0.0524 0.1659       0.0176
    8 0.6489  0.0813 0.2698       0.0172
    9 0.7483  0.1762 0.0755       0.0172
   10 0.7484  0.1775 0.0742       0.0172

Pesos óptimos finales: w_rp3=0.795  w_ease=0.067  w_bpr=0.138
NDCG@10 val: 0.01779


In [16]:
# ── Lever D — Evaluación FINAL en test (UNA SOLA VEZ) ────────────────────────
print(f"Evaluando ensemble FINAL en test ({len(test_users_b)})...")
t0 = time.time()
get_final_ens = make_final_ensemble(final_w1, final_w2, W_rp3_final, B_ease_opt, X_rp3_final)
m_final_test = evaluate(
    get_final_ens, test_users_b,
    test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f"  [{time.time()-t0:.1f}s]")

ndcg_final_test = m_final_test["NDCG@10"]
delta_d = (ndcg_final_test - NDCG_NB11_ENS2) / NDCG_NB11_ENS2 * 100
TARGET = 0.030

print(f"\nLEVER D — Resultados test:")
print(f"  Ensemble Final : NDCG@10={ndcg_final_test:.5f}")
print(f"  NB11 Ens2      : NDCG@10={NDCG_NB11_ENS2:.5f}")
print(f"  Mejora         : {'+' if delta_d>=0 else ''}{delta_d:.1f}%")
print(f"  Target (0.030) : {'✅ SUPERADO' if ndcg_final_test >= TARGET else '❌ No alcanzado'}")

Evaluando ensemble FINAL en test (2551)...
  [343.6s]

LEVER D — Resultados test:
  Ensemble Final : NDCG@10=0.02791
  NB11 Ens2      : NDCG@10=0.02603
  Mejora         : +7.3%
  Target (0.030) : ❌ No alcanzado


## E — Tabla comparativa NB13 y actualización de artefactos

In [17]:
# ── Compilar tabla comparativa ────────────────────────────────────────────────
summary_rows = [
    {"Notebook": "NB11",   "Modelo": f"EASE^R (λ={EASE_REG_BASE}) [ref]",
     "NDCG@10": NDCG_EASE_BASE_NB11,
     "Prec@10": float("nan"),
     "Recall@10": float("nan"),
     "Coverage": float("nan"),
     "Novelty":  float("nan")},
    {"Notebook": "NB11",   "Modelo": "RP3beta opt (α=0.75,β=0.30)",
     "NDCG@10": ndcg_rp3opt_test,
     "Prec@10": m_rp3opt_test.get("Precision@10", float("nan")),
     "Recall@10": m_rp3opt_test.get("Recall@10", float("nan")),
     "Coverage": m_rp3opt_test.get("Coverage", float("nan")),
     "Novelty":  m_rp3opt_test.get("Novelty",  float("nan"))},
    {"Notebook": "NB13-A", "Modelo": f"EASE^R opt (λ={best_lambda:.1f})",
     "NDCG@10": ndcg_ease_opt_test,
     "Prec@10": m_ease_opt_test.get("Precision@10", float("nan")),
     "Recall@10": m_ease_opt_test.get("Recall@10", float("nan")),
     "Coverage": m_ease_opt_test.get("Coverage", float("nan")),
     "Novelty":  m_ease_opt_test.get("Novelty",  float("nan"))},
    {"Notebook": "NB13-B", "Modelo": f"Ens3-way RP3+EASE+BPR (w1={best_w1:.2f},w2={best_w2:.2f},w3={best_w3:.2f})",
     "NDCG@10": ndcg_ens3_test,
     "Prec@10": m_ens3_test.get("Precision@10", float("nan")),
     "Recall@10": m_ens3_test.get("Recall@10", float("nan")),
     "Coverage": m_ens3_test.get("Coverage", float("nan")),
     "Novelty":  m_ens3_test.get("Novelty",  float("nan"))},
    {"Notebook": "NB13-C", "Modelo": f"RP3+TD (decay={best_decay_rate})",
     "NDCG@10": ndcg_rp3_td_test,
     "Prec@10": m_rp3_td_test.get("Precision@10", float("nan")),
     "Recall@10": m_rp3_td_test.get("Recall@10", float("nan")),
     "Coverage": m_rp3_td_test.get("Coverage", float("nan")),
     "Novelty":  m_rp3_td_test.get("Novelty",  float("nan"))},
    {"Notebook": "NB13-D", "Modelo": f"EnsembleFinal ({rp3_label}+EASE+BPR)",
     "NDCG@10": ndcg_final_test,
     "Prec@10": m_final_test.get("Precision@10", float("nan")),
     "Recall@10": m_final_test.get("Recall@10", float("nan")),
     "Coverage": m_final_test.get("Coverage", float("nan")),
     "Novelty":  m_final_test.get("Novelty",  float("nan"))},
]

df_nb13 = pd.DataFrame(summary_rows)
for col in ["NDCG@10", "Prec@10", "Recall@10"]:
    df_nb13[col] = df_nb13[col].round(6)
df_nb13["Coverage"] = df_nb13["Coverage"].round(4)
df_nb13["Novelty"]  = df_nb13["Novelty"].round(2)

print("\n" + "=" * 95)
print("TABLA NB13 — Comparativa de modelos (test set)")
print("=" * 95)
print(df_nb13.to_string(index=False))
print()

out_csv = DATA_DIR / "model_comparison_nb13.csv"
df_nb13.to_csv(out_csv, index=False)
print(f"CSV guardado: {out_csv}")



TABLA NB13 — Comparativa de modelos (test set)
Notebook                                          Modelo  NDCG@10  Prec@10  Recall@10  Coverage  Novelty
    NB11                          EASE^R (λ=500.0) [ref] 0.024766      NaN        NaN       NaN      NaN
    NB11                     RP3beta opt (α=0.75,β=0.30) 0.025835 0.006037   0.038064    0.0572    15.20
  NB13-A                             EASE^R opt (λ=14.3) 0.020909 0.005410   0.033324    0.0514    15.10
  NB13-B Ens3-way RP3+EASE+BPR (w1=0.71,w2=0.08,w3=0.21) 0.024980 0.006037   0.037933    0.0711    15.31
  NB13-C                             RP3+TD (decay=0.01) 0.028586 0.006821   0.041412    0.0523    15.10
  NB13-D         EnsembleFinal (RP3+TD(d=0.01)+EASE+BPR) 0.027914 0.006821   0.040699    0.0677    15.27

CSV guardado: C:\Users\Usuario\Desktop\HENRY\PROYECTO FINAL\EDA PRUEBA\nexus-recsys\data\processed\model_comparison_nb13.csv


In [18]:
# ── Actualizar model_comparison_final.csv si hay nuevo campeón ───────────────
final_csv = DATA_DIR / "model_comparison_final.csv"
df_final  = pd.read_csv(final_csv)

# El mejor modelo NB13
best_nb13_idx   = df_nb13["NDCG@10"].idxmax()
best_nb13_row   = df_nb13.loc[best_nb13_idx]
best_nb13_ndcg  = best_nb13_row["NDCG@10"]
current_best_ndcg = df_final["NDCG@10"].max() if "NDCG@10" in df_final.columns else (
    df_final.iloc[:, 1].max()  # fallback by position
)

print(f"Mejor NB13: {best_nb13_row['Modelo']}  NDCG@10={best_nb13_ndcg:.5f}")
print(f"Actual campeón : NDCG@10={current_best_ndcg:.5f}")

if best_nb13_ndcg > current_best_ndcg:
    new_row = pd.DataFrame([{
        col: best_nb13_row.get(col, None) for col in df_final.columns
        if col in best_nb13_row.index
    }])
    # Añadir columnas faltantes
    for col in df_final.columns:
        if col not in new_row.columns:
            new_row[col] = None
    new_row["Notebook"] = best_nb13_row["Notebook"] if "Notebook" in best_nb13_row.index else "NB13"
    new_row["Modelo"]   = best_nb13_row["Modelo"]
    df_final = pd.concat([df_final, new_row], ignore_index=True)
    df_final.to_csv(final_csv, index=False)
    print(f"✅ model_comparison_final.csv actualizado con nuevo campeón (NDCG@10={best_nb13_ndcg:.5f})")
else:
    print(f"ℹ️  NB11 sigue siendo campeón (NDCG@10={current_best_ndcg:.5f}), no se actualiza")

Mejor NB13: RP3+TD (decay=0.01)  NDCG@10=0.02859
Actual campeón : NDCG@10=0.02603
✅ model_comparison_final.csv actualizado con nuevo campeón (NDCG@10=0.02859)


In [19]:
# ── Actualizar docs/model_justification.md ────────────────────────────────────
import re

doc_path = DOCS_DIR / "model_justification.md"
doc_text = doc_path.read_text(encoding="utf-8") if doc_path.exists() else ""

NDCG_NB11_ENS2_d = 0.026026

section = f"""

---

## 5.6 — NB13: Ensemble Avanzado (4 Levers hacia NDCG@10 ≥ 0.030)

**Objetivo:** Superar NDCG@10 = 0.026026 (NB11) apuntando a ≥ 0.030 (+15.2%).  
**Dataset:** RetailRocket — cutoff 2015-08-22 — 3 000 warm users (seed=42).  
**Tercer componente:** BPR-MF (64 dims) de `final_model_v3.pkl`; MultiVAE no disponible como artefacto independiente.

### Lever A — EASE^R λ Optuna
- λ_base = 500.0 (NB11, nunca optimizado)
- Búsqueda log-scale [10, 5000], {N_TRIALS_EASE} trials, G = X^T X precomputado
- **λ_óptimo = {best_lambda:.1f}**  NDCG@10 val = {best_ease_val:.5f}
- **Test: {ndcg_ease_opt_test:.5f}** vs base {ndcg_ease_base_test:.5f}  ({'+' if ndcg_ease_opt_test>=ndcg_ease_base_test else ''}{(ndcg_ease_opt_test-ndcg_ease_base_test)/ndcg_ease_base_test*100:.1f}%)

### Lever B — Ensemble 3-way Optuna (RP3 + EASE + BPR)
- Pesos Optuna {N_TRIALS_ENSEMBLE} trials: w_rp3∈[0.30,0.80], w_ease∈[0.05,0.40], w_bpr=1-w1-w2
- **Pesos óptimos: w_rp3={best_w1:.3f}  w_ease={best_w2:.3f}  w_bpr={best_w3:.3f}**
- **Test NDCG@10 = {ndcg_ens3_test:.5f}** ({'+' if ndcg_ens3_test>=NDCG_NB11_ENS2_d else ''}{(ndcg_ens3_test-NDCG_NB11_ENS2_d)/NDCG_NB11_ENS2_d*100:.1f}% vs NB11)

### Lever C — Temporal Decay RP3beta
- Pesos recencia: exp(-decay_rate × días_antes_cutoff)  grid: [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
- **Mejor decay_rate = {best_decay_rate}**  NDCG@10 val = {best_decay_val:.5f}
- **Test: RP3+TD = {ndcg_rp3_td_test:.5f}** vs RP3opt = {ndcg_rp3opt_test:.5f}  ({'+' if delta_c>=0 else ''}{delta_c:.1f}%)
- Componente RP3 para Lever D: **{rp3_label}**

### Lever D — Ensemble Final ({rp3_label} + EASE + BPR)
- {N_TRIALS_FINAL} trials Optuna sobre pesos de los 3 componentes
- **Pesos finales: w_rp3={final_w1:.3f}  w_ease={final_w2:.3f}  w_bpr={final_w3:.3f}**
- **Test NDCG@10 = {ndcg_final_test:.5f}**  ({'+' if delta_d>=0 else ''}{delta_d:.1f}% vs NB11 0.026026)
- Target 0.030: **{'✅ SUPERADO' if ndcg_final_test >= 0.030 else '❌ No alcanzado'}**

### Tabla resumen NB13
| Notebook | Modelo | NDCG@10 | Prec@10 |
|---|---|---|---|
"""

for _, row in df_nb13.iterrows():
    section += f"| {row['Notebook']} | {row['Modelo']} | {row['NDCG@10']:.5f} | {row['Prec@10']:.5f} |\n"

section += "\nArtefactos: `encoders/ease_optimized.pkl`, `data/processed/model_comparison_nb13.csv`\n"

if "## 5.6" in doc_text:
    doc_text = re.sub(r"\n---\n\n## 5\.6.*", section, doc_text, flags=re.DOTALL)
else:
    doc_text += section

doc_path.write_text(doc_text, encoding="utf-8")
print(f"docs/model_justification.md actualizado — sección 5.6 añadida/actualizada.")


docs/model_justification.md actualizado — sección 5.6 añadida/actualizada.


In [ ]:
# ── Reporte final ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("NB13 — REPORTE FINAL")
print("=" * 70)
print(f"Punto de partida (NB11 Ens2): NDCG@10 = {NDCG_NB11_ENS2:.5f}")
print(f"Target                      : NDCG@10 = 0.03000")
print()
print(f"Lever A  EASE λ opt ({best_lambda:.1f})  : {ndcg_ease_opt_test:.5f}  ({'+' if ndcg_ease_opt_test>=NDCG_NB11_ENS2 else ''}{(ndcg_ease_opt_test-NDCG_NB11_ENS2)/NDCG_NB11_ENS2*100:.1f}%)")
print(f"Lever B  3-way Ens opt       : {ndcg_ens3_test:.5f}  ({'+' if ndcg_ens3_test>=NDCG_NB11_ENS2 else ''}{(ndcg_ens3_test-NDCG_NB11_ENS2)/NDCG_NB11_ENS2*100:.1f}%)")
print(f"Lever C  RP3+TD (d={best_decay_rate})   : {ndcg_rp3_td_test:.5f}  ({'+' if ndcg_rp3_td_test>=NDCG_NB11_ENS2 else ''}{(ndcg_rp3_td_test-NDCG_NB11_ENS2)/NDCG_NB11_ENS2*100:.1f}%)")
print(f"Lever D  Ensemble Final      : {ndcg_final_test:.5f}  ({'+' if ndcg_final_test>=NDCG_NB11_ENS2 else ''}{(ndcg_final_test-NDCG_NB11_ENS2)/NDCG_NB11_ENS2*100:.1f}%)")
print()
best_nb13_overall = max(ndcg_ease_opt_test, ndcg_ens3_test, ndcg_rp3_td_test, ndcg_final_test)
print(f"MEJOR RESULTADO NB13         : NDCG@10 = {best_nb13_overall:.5f}")
print(f"Target 0.030{'   ✅ SUPERADO' if best_nb13_overall >= 0.030 else '   ❌ No alcanzado'}")
print("=" * 70)


NB13 — REPORTE FINAL
Punto de partida (NB11 Ens2): NDCG@10 = 0.02603
Target                      : NDCG@10 = 0.03000

Lever A  EASE λ opt (14.3)  : 0.02091  (-19.7%)
Lever B  3-way Ens opt       : 0.02498  (-4.0%)
Lever C  RP3+TD (d=0.01)   : 0.02859  (+9.8%)
Lever D  Ensemble Final      : 0.02791  (+7.3%)

MEJOR RESULTADO NB13         : NDCG@10 = 0.02859
Target 0.030   ❌ No alcanzado


: 